<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeshengliu213/ai-for-materials/blob/main/course/07%20-%20Neural%20Networks%20and%20CNNs/solutions/07_nn_solutions.ipynb)

# 第七讲随堂练习参考答案（`07a` – `07b`）

本 notebook 汇总了 `07a_neural_networks.ipynb` 与 `07b_cnn_microstructure.ipynb` 中全部 6 道随堂练习（`07a-1` 至 `07b-3`）的参考实现与结果分析。

In [ ]:
import copy
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors

%matplotlib inline
plt.rcParams["figure.dpi"] = 110
RDLogger.DisableLog("rdApp.*")

def set_seed(seed=0):
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.use_deterministic_algorithms(True)

set_seed(0)

<!-- ai4m-style -->
> **📘 课程统一绘图风格**：下面这格加载课程自带的 `ai4m.mplstyle`（配色与课程网站同源，无上右边框，仅横向浅网格，保存默认 300 dpi）。`C` 是课程配色，手动指定颜色时写 `color=C.crimson`。


In [ ]:
# ai4m-style：课程统一绘图风格（本地与 Colab 通用）
import pathlib, sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))   # course/ 根目录
import ai4m_style as ai4m

C = ai4m.use()        # 应用 ai4m.mplstyle，返回课程配色


---
## 07a · 神经网络基础与多层感知机

先把 `07a` 共用的数据集划分与训练辅助函数准备好：

In [ ]:
DATA_FILE = "../data/solubility.csv"
if not os.path.exists(DATA_FILE):
    DATA_FILE = "data/solubility.csv"

df = pd.read_csv(DATA_FILE)
df["mol"] = df["SMILES"].apply(Chem.MolFromSmiles)
df = df[df["mol"].notna()].reset_index(drop=True)

feature_cols = [
    "MolWt", "MolLogP", "MolMR", "HeavyAtomCount", "NumHAcceptors", "NumHDonors",
    "NumHeteroatoms", "NumRotatableBonds", "NumValenceElectrons", "NumAromaticRings",
    "NumSaturatedRings", "NumAliphaticRings", "RingCount", "TPSA", "LabuteASA",
    "BalabanJ", "BertzCT",
]
X = df[feature_cols].values
y = df["Solubility"].values

X_trval, X_test, y_trval, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(X_trval, y_trval, test_size=0.1, random_state=42)

scaler = StandardScaler().fit(X_train)
to_tensor = lambda a: torch.tensor(a, dtype=torch.float32)
Xtr, Xva, Xte = (to_tensor(scaler.transform(a)) for a in (X_train, X_valid, X_test))
ytr, yva, yte = (to_tensor(a) for a in (y_train, y_valid, y_test))
train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=64, shuffle=True)

loss_fn = nn.MSELoss()

def rmse(model, X_t, y_t):
    model.eval()
    with torch.no_grad():
        return torch.sqrt(loss_fn(model(X_t), y_t)).item()

def fit(model, n_epochs=100, lr=1e-3, weight_decay=0.0):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    history = {"train": [], "valid": []}
    best_valid, best_state = float("inf"), None
    for _ in range(n_epochs):
        model.train()
        for xb, yb in train_loader:
            loss = loss_fn(model(xb), yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        history["train"].append(rmse(model, Xtr, ytr))
        history["valid"].append(rmse(model, Xva, yva))
        if history["valid"][-1] < best_valid:
            best_valid = history["valid"][-1]
            best_state = copy.deepcopy(model.state_dict())
    return history, best_state

### 练习 07a-1：扩展自定义类的方法

In [ ]:
class Molecule:
    def __init__(self, name, smiles):
        self.name = name
        self.smiles = smiles
        self.mol = Chem.MolFromSmiles(smiles)

    def n_heavy_atoms(self):
        return self.mol.GetNumHeavyAtoms()

    def n_aromatic_rings(self):
        return Descriptors.NumAromaticRings(self.mol)

    def describe(self):
        return (
            f"{self.name}: {self.n_heavy_atoms()} heavy atoms, "
            f"{self.n_aromatic_rings()} aromatic rings"
        )

for m in [
    Molecule("thiophene", "c1ccsc1"),
    Molecule("pentacene", "c1ccc2cc3cc4cc5ccccc5cc4cc3cc2c1"),
]:
    print(m.describe())

### 练习 07a-2：对比不同非线性激活函数（ReLU vs. Tanh）

In [ ]:
class FlexibleMLP(nn.Module):
    def __init__(self, n_features, hidden_size=64, activation=nn.ReLU):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden_size),
            activation(),
            nn.Linear(hidden_size, hidden_size),
            activation(),
            nn.Linear(hidden_size, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)

for act in [nn.ReLU, nn.Tanh]:
    set_seed(0)
    m = FlexibleMLP(17, hidden_size=64, activation=act)
    hist, best_st = fit(m, n_epochs=100)
    last_rmse = rmse(m, Xte, yte)
    m.load_state_dict(best_st)
    best_rmse = rmse(m, Xte, yte)
    print(f"{act.__name__:5s} -> 最后一轮测试 RMSE: {last_rmse:.2f} | 早停后测试 RMSE: {best_rmse:.2f}")

**结果分析**：在仅有两层隐藏层的浅层网络上，`Tanh` 与 `ReLU` 的测试 RMSE 都处在 $1.17\sim 1.23$ 之间，差距不大。只有当网络层数加深到十余层以上时，`Tanh` 两端饱和区导数趋近于 0 所引起的梯度消失才会显著拖慢收敛，此时 `ReLU` 或 `GELU` 的优势才会充分体现。

### 练习 07a-3：将 MLP 改造为水溶解度二分类器

In [ ]:
y_cls_tr, y_cls_va, y_cls_te = (to_tensor((a < -4).astype(float)) for a in (y_train, y_valid, y_test))
cls_loader = DataLoader(TensorDataset(Xtr, y_cls_tr), batch_size=64, shuffle=True)
bce = nn.BCEWithLogitsLoss()

def predict_proba(model, X_t):
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(X_t)).numpy()

set_seed(0)
clf = FlexibleMLP(n_features=17, hidden_size=64, activation=nn.ReLU)
optimizer = torch.optim.Adam(clf.parameters(), lr=1e-3)
best_auc, best_state = 0.0, None

for epoch in range(50):
    clf.train()
    for xb, yb in cls_loader:
        loss = bce(clf(xb), yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    va_auc = roc_auc_score(y_cls_va.numpy(), predict_proba(clf, Xva))
    if va_auc > best_auc:
        best_auc = va_auc
        best_state = copy.deepcopy(clf.state_dict())

clf.load_state_dict(best_state)
te_auc = roc_auc_score(y_cls_te.numpy(), predict_proba(clf, Xte))
print(f"验证集最高 ROC-AUC: {best_auc:.3f}")
print(f"测试集 ROC-AUC:     {te_auc:.3f} (对照第 02c 讲随机森林: 0.953)")

---
## 07b · 卷积神经网络与材料显微图像分析

### 练习 07b-1：计算卷积层参数量与输出空间尺寸

In [ ]:
x_batch = torch.zeros(8, 1, 64, 64)
conv = nn.Conv2d(in_channels=1, out_channels=24, kernel_size=5, stride=2, padding=0)
out = conv(x_batch)

# 手算公式验证:
# H_out = floor((64 + 2*0 - 5) / 2) + 1 = floor(59 / 2) + 1 = 29 + 1 = 30
# 参数量 = 24 * (1 * 5 * 5 + 1) = 24 * 26 = 624
n_params = sum(p.numel() for p in conv.parameters())
print("Output tensor shape:", out.shape, "(Expected: [8, 24, 30, 30])")
print("Total parameters:   ", n_params, "(Expected: 624)")

### 练习 07b-2：实现旋转与翻转不变的数据增强函数

In [ ]:
def augment_batch(xb, rng=None):
    if rng is None:
        rng = np.random.RandomState()
    k = int(rng.randint(0, 4))
    out = torch.rot90(xb, k=k, dims=(2, 3))
    if rng.rand() < 0.5:
        out = torch.flip(out, dims=(3,))
    return out

# 生成一张层片状珠光体示例图并展示 4 次随机增强结果
yy, xx = np.mgrid[0:48, 0:48].astype(np.float32) / 48.0
theta = np.pi / 5
wave = np.sin(2 * np.pi * 6.5 * (xx * np.cos(theta) + yy * np.sin(theta)))
pearlite = torch.tensor(((wave > 0).astype(np.float32) * 0.7 + 0.15))[None, None, :, :]

rng_aug = np.random.RandomState(7)
fig, axes = plt.subplots(1, 5, figsize=(11, 2.5))
axes[0].imshow(pearlite[0, 0], cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Original")
axes[0].axis("off")
for i in range(1, 5):
    aug_img = augment_batch(pearlite, rng=rng_aug)
    axes[i].imshow(aug_img[0, 0], cmap="gray", vmin=0, vmax=1)
    axes[i].set_title(f"Augmented #{i}")
    axes[i].axis("off")
plt.tight_layout()
plt.show()

### 练习 07b-3：用 CNN 端到端回归球状析出相面积分数

In [ ]:
class MicrostructureCNN(nn.Module):
    def __init__(self, n_outputs=1):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.head = nn.Linear(32, n_outputs)

    def forward(self, x):
        return self.head(self.features(x).flatten(1)).squeeze(1)

def make_precipitate_area_dataset(n_samples=240, size=48, seed=123):
    rng = np.random.RandomState(seed)
    yy, xx = np.mgrid[0:size, 0:size].astype(np.float32) / size
    imgs, area_fracs = [], []
    for _ in range(n_samples):
        mask = np.zeros((size, size), dtype=bool)
        n_blobs = rng.randint(2, 16)
        for _ in range(n_blobs):
            cy, cx = rng.uniform(0.1, 0.9, size=2)
            r = rng.uniform(0.05, 0.14)
            mask |= ((yy - cy) ** 2 + (xx - cx) ** 2 <= r ** 2)
        area_fracs.append(float(mask.mean()))
        img = np.where(mask, 0.85, 0.18).astype(np.float32)
        img = gaussian_filter(img, sigma=0.7) + rng.normal(0, 0.05, (size, size)).astype(np.float32)
        imgs.append(np.clip(img, 0.0, 1.0))
    return torch.tensor(np.array(imgs)[:, None, :, :]), torch.tensor(area_fracs, dtype=torch.float32)

X_af, y_af = make_precipitate_area_dataset()
X_af_tr, X_af_te = X_af[:180], X_af[180:]
y_af_tr, y_af_te = y_af[:180], y_af[180:]

set_seed(42)
reg_cnn = MicrostructureCNN(n_outputs=1)
af_loader = DataLoader(TensorDataset(X_af_tr, y_af_tr), batch_size=32, shuffle=True)
opt_af = torch.optim.Adam(reg_cnn.parameters(), lr=3e-3)
mse_af = nn.MSELoss()

for epoch in range(30):
    reg_cnn.train()
    for xb, yb in af_loader:
        loss = mse_af(reg_cnn(xb), yb)
        opt_af.zero_grad()
        loss.backward()
        opt_af.step()

reg_cnn.eval()
with torch.no_grad():
    pred_af = reg_cnn(X_af_te).numpy()
true_af = y_af_te.numpy()
mae = np.mean(np.abs(pred_af - true_af))
print(f"Test Set Precipitate Area Fraction MAE: {mae:.4f}")

fig, ax = plt.subplots(figsize=(4.6, 4.6))
ax.scatter(true_af, pred_af, s=28, alpha=0.75, color="#1f77b4")
ax.plot([0, 0.55], [0, 0.55], "k--", lw=1)
ax.set_xlabel("True Precipitate Area Fraction")
ax.set_ylabel("Predicted Area Fraction (CNN)")
ax.set_title(f"CNN Area Fraction Regression (MAE = {mae:.3f})")
plt.tight_layout()
plt.show()